# SET OS · EVA Stage-1 encoder warm-start

This notebook is orchestration only. It uploads one deterministic bundle, acquires the pinned official EVA source, builds the existing silver projection, and invokes `pretrain_eva.py`. The run trains only `CandidateA.full_frame_backbone` with disposable EVA-native auxiliary heads. It is research-only, non-gold, not release-admissible, and does not select or export a Camera Coach runtime model.

EVA is AVA-derived; the upstream repository declaration does not resolve the underlying image rights. Keep the downloaded data and all checkpoints outside Git.

In [ ]:
from pathlib import Path
import hashlib
import json
import re
import shutil
import stat
import subprocess
import sys
import zipfile
from IPython.display import display

WORK_ROOT = Path('/content/setos_eva_stage1')
BUNDLE_PATH = WORK_ROOT / 'SET_OS_EVA_STAGE1.zip'
EXTRACT_ROOT = WORK_ROOT / 'bundle'
# Paste the SHA-256 printed by package_camera_colab.py before extraction.
EXPECTED_BUNDLE_SHA256 = ''
EXPECTED_BUNDLE_FILES = [
    'datasets/camera-coach/v1/sources/eva-fb40a9f1.json',
    'ml/camera_coach/configs/eva_stage1_colab.json',
    'ml/camera_coach/contracts/set_composition_net_v1.json',
    'ml/camera_coach/colab/SET_OS_EVA_STAGE1.ipynb',
    'ml/camera_coach/models/__init__.py',
    'ml/camera_coach/models/set_composition_net.py',
    'ml/camera_coach/pretrain_eva.py',
    'tools/dataset/build_eva_silver.py',
    'tools/dataset/camera_source_intake.py',
    'tools/dataset/fetch_eva.py',
]
BUNDLE_MANIFEST_SCHEMA = 'camera-eva-stage1-colab-bundle-v1'
BUNDLE_MANIFEST_VERSION = '1.0.0'
MAX_MEMBER_BYTES = 2 * 1024 * 1024
MAX_TOTAL_BYTES = 8 * 1024 * 1024
USE_DRIVE = False
RESUME = False
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print('Set EXPECTED_BUNDLE_SHA256, USE_DRIVE, and RESUME above, then run the cells in order.')

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/SET_OS/EVA_STAGE1')
    WORK_ROOT = DRIVE_ROOT / 'workspace'
    BUNDLE_PATH = WORK_ROOT / 'SET_OS_EVA_STAGE1.zip'
    EXTRACT_ROOT = WORK_ROOT / 'bundle'
    DATA_ROOT = DRIVE_ROOT / 'data'
    # Choose the persistent run root before the trainer starts. A runtime
    # loss therefore leaves receipt/checkpoints available for RESUME=True.
    RUN_ROOT = DRIVE_ROOT / 'run'
    DRIVE_OUTPUT = DRIVE_ROOT / 'exports'
else:
    DRIVE_ROOT = None
    DATA_ROOT = Path('/content/setos_eva_data')
    RUN_ROOT = Path('/content/setos_eva_stage1_run')
    DRIVE_OUTPUT = None
RUN_ROOT.parent.mkdir(parents=True, exist_ok=True)
DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print('persistent run root:', RUN_ROOT, 'drive export:', DRIVE_OUTPUT)

In [ ]:
# One upload: the deterministic bundle generated by package_camera_colab.py.
if not BUNDLE_PATH.is_file():
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError(f'expected exactly one bundle upload, got {len(uploaded)}')
    uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
    if not uploaded_name.lower().endswith('.zip'):
        raise RuntimeError('uploaded file must be a .zip bundle')
    BUNDLE_PATH.write_bytes(uploaded_bytes)
print(BUNDLE_PATH, BUNDLE_PATH.stat().st_size, 'bytes')

In [ ]:
# Verify the user-supplied whole-bundle trust anchor before extraction.
if not re.fullmatch(r'[0-9a-f]{64}', EXPECTED_BUNDLE_SHA256):
    raise RuntimeError('paste the lowercase SHA-256 printed by package_camera_colab.py into EXPECTED_BUNDLE_SHA256')
actual_bundle_sha256 = hashlib.sha256(BUNDLE_PATH.read_bytes()).hexdigest()
if actual_bundle_sha256 != EXPECTED_BUNDLE_SHA256:
    raise RuntimeError(f'whole bundle SHA-256 mismatch: expected {EXPECTED_BUNDLE_SHA256}, got {actual_bundle_sha256}')
if BUNDLE_PATH.stat().st_size > MAX_TOTAL_BYTES:
    raise RuntimeError('bundle exceeds the total size ceiling')
# Embedded manifests are secondary integrity data, never the trust anchor.
if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)
EXTRACT_ROOT.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE_PATH) as archive:
    manifest = json.loads(archive.read('bundle-manifest.json'))
    sums = json.loads(archive.read('SHA256SUMS.json'))
    if set(manifest) != {'schema_id', 'schema_version', 'kind', 'files', 'disclaimer'}:
        raise RuntimeError('bundle manifest schema keys drifted')
    if manifest['schema_id'] != BUNDLE_MANIFEST_SCHEMA or manifest['schema_version'] != BUNDLE_MANIFEST_VERSION or manifest['kind'] != 'research_only_colab_orchestration':
        raise RuntimeError('bundle manifest schema/version/kind is unsupported')
    records = manifest['files']
    if not isinstance(records, list) or [record.get('path') for record in records] != EXPECTED_BUNDLE_FILES:
        raise RuntimeError('bundle source allowlist mismatch')
    record_keys = {'path', 'bytes', 'sha256'}
    if any(not isinstance(record, dict) or set(record) != record_keys or not isinstance(record['bytes'], int) or record['bytes'] < 0 or not re.fullmatch(r'[0-9a-f]{64}', record['sha256']) for record in records):
        raise RuntimeError('bundle file manifest record schema is invalid')
    if set(sums) != {'schema_id', 'files'} or sums['schema_id'] != 'camera-eva-stage1-sha256-v1' or sums.get('files') != records:
        raise RuntimeError('bundle SHA-256 manifest schema or records mismatch')
    expected = [*EXPECTED_BUNDLE_FILES, 'bundle-manifest.json', 'SHA256SUMS.json']
    if archive.namelist() != expected:
        raise RuntimeError('bundle member order/allowlist mismatch')
    total_uncompressed = sum(info.file_size for info in archive.infolist())
    if total_uncompressed > MAX_TOTAL_BYTES:
        raise RuntimeError('bundle members exceed the total uncompressed size ceiling')
    for info in archive.infolist():
        if info.file_size > MAX_MEMBER_BYTES:
            raise RuntimeError(f'bundle member exceeds size ceiling: {info.filename}')
        relative = Path(info.filename)
        if relative.is_absolute() or '..' in relative.parts or info.filename.startswith('/'):
            raise RuntimeError(f'unsafe ZIP path: {info.filename}')
        mode = (info.external_attr >> 16) & 0o170000
        if mode == stat.S_IFLNK:
            raise RuntimeError(f'symlink in bundle: {info.filename}')
        if mode != stat.S_IFREG:
            raise RuntimeError(f'non-regular bundle member: {info.filename}')
        target = EXTRACT_ROOT.joinpath(*relative.parts)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(info.filename))
    for record in records:
        payload = (EXTRACT_ROOT / record['path']).read_bytes()
        if len(payload) != record['bytes'] or hashlib.sha256(payload).hexdigest() != record['sha256']:
            raise RuntimeError(f'bundle hash mismatch: {record["path"]}')
print('verified whole bundle', actual_bundle_sha256, 'and', len(records), 'allowlisted source files')

In [ ]:
# Colab's Linux runtime uses its installed Torch when compatible. The notebook
# records actual versions and does not apply the macOS requirements.lock.
import torch
from PIL import Image
print({'python': sys.version.split()[0], 'torch': torch.__version__, 'cuda': torch.cuda.is_available(), 'pillow': getattr(Image, '__version__', 'unknown'), 'bundle_sha256': actual_bundle_sha256})

In [ ]:
# Acquire the pinned EVA source using the existing repository fetcher.
fetcher = EXTRACT_ROOT / 'tools/dataset/fetch_eva.py'
if not (DATA_ROOT / 'receipts/eva-source-receipt.json').is_file():
    subprocess.run([sys.executable, str(fetcher), 'fetch', '--data-root', str(DATA_ROOT)], check=True)
else:
    print('Existing external EVA receipt found; fetcher will be reused on demand.')

In [ ]:
# Project upstream votes into the existing research-only silver manifest.
silver_builder = EXTRACT_ROOT / 'tools/dataset/build_eva_silver.py'
subprocess.run([sys.executable, str(silver_builder), '--data-root', str(DATA_ROOT)], check=True)
silver_receipt = json.loads((DATA_ROOT / 'receipts/eva-silver-receipt.json').read_text())
display({'records': silver_receipt['output']['records'], 'manifest_sha256': silver_receipt['output']['sha256'], 'research_only': silver_receipt['release_admissible'] is False})

In [ ]:
# Run or resume the single source-of-truth trainer. No trainer logic is copied here.
trainer = EXTRACT_ROOT / 'ml/camera_coach/pretrain_eva.py'
config = EXTRACT_ROOT / 'ml/camera_coach/configs/eva_stage1_colab.json'
command = [sys.executable, str(trainer), '--config', str(config), '--data-root', str(DATA_ROOT), '--run-dir', str(RUN_ROOT), '--device', 'auto']
if RESUME:
    command += ['--resume', 'latest']
subprocess.run(command, check=True)

In [ ]:
receipt = json.loads((RUN_ROOT / 'receipt.json').read_text())
metrics = (RUN_ROOT / 'metrics.jsonl').read_text().splitlines()
display({'status': receipt['status'], 'counts': receipt['counts'], 'fit': receipt['fit'], 'preprocessing': receipt['preprocessing'], 'data_hash': receipt['data_hash'], 'model_contract_sha256': receipt['model_contract_sha256'], 'contract_source_sha256': receipt['contract_source_sha256'], 'last_checkpoint': receipt['last_checkpoint'], 'final_artifact': receipt.get('final_artifact')})
if receipt['status'] == 'complete':
    print('encoder-final.pt sha256:', receipt['final_artifact']['sha256'])
    print('metrics rows:', len(metrics))
else:
    print('Run is resumable; set RESUME=True and rerun the trainer cell.')

In [ ]:
if receipt['status'] == 'complete':
    if DRIVE_OUTPUT is not None:
        DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
        shutil.copytree(RUN_ROOT, DRIVE_OUTPUT, dirs_exist_ok=True)
        print('copied run artifacts to', DRIVE_OUTPUT)
    from google.colab import files
    files.download(str(RUN_ROOT / 'encoder-final.pt'))
else:
    print('No final artifact to download until the configured epochs finish.')

## Boundary

`encoder-final.pt` is an encoder-only, disposable research warm-start. It contains no EVA auxiliary heads and no SET runtime heads. It is not a production candidate, does not prove Camera Coach quality, and must remain separate from the later human-gold M3 training/evaluation chain.